In [0]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus as urlquote, quote
engine = create_engine('mysql+pymysql://admin:{}@database-2.cv0g6mqwqu0z.ap-south-1.rds.amazonaws.com/employee'.format(quote('Root#123')), echo=False)

In [0]:
from sqlalchemy import text


table_found = False
max_timestamp = ''
table_name_table_info = "employee.tables_info"
statement = text("SELECT * from {} where table_name = 'customer'".format(table_name_table_info))
print(statement)

with engine.connect() as connection:
    result = connection.execute(statement)
    results = result.fetchall()
    if len(results) > 0:
        print('Table found')
        table_found = True
        max_timestamp = results[0][1]
        print(max_timestamp)
    else:
        insert_statement = f"INSERT INTO {table_name_table_info} VALUES('customer','{max_timestamp}')"
        print(insert_statement)
        connection.execute(text(insert_statement))
        connection.commit()
        print('Table not found')

In [0]:
resut_table_name = 'employee.customer'

In [0]:
with engine.connect() as connection:
    if table_found:
        result = connection.execute(text("SELECT min(id),max(id) from {} WHERE updateon > '{}';".format(resut_table_name,max_timestamp)))
        rows = result.fetchall()
        for row in rows:
            lower_bound = row[0]
            upper_bound = row[1]
    else:
        result = connection.execute(text("SELECT min(id),max(id) from {} ;".format(resut_table_name,max_timestamp)))
        rows = result.fetchall()
        for row in rows:
            lower_bound = row[0]
            upper_bound = row[1]

print('lowerbound:{},upperbound:{}'.format(lower_bound,upper_bound))

In [0]:
connection_details = {
    "user": "admin",
    "password": "Root#123",
    "driver": "com.mysql.cj.jdbc.Driver",
    'partitionColumn': "Id",
    'lowerBound': str(lower_bound),
    'upperBound': str(upper_bound),
    'numPartitions': "4"
}

if table_found:
    select_query = f"(select * from customer WHERE updateon > '{max_timestamp}' ) as foo"
else:
    select_query = f"(select * from customer) as foo"


print(select_query)

jdbc_url = 'jdbc:mysql://database-2.cv0g6mqwqu0z.ap-south-1.rds.amazonaws.com:3306/employee'

source_df = spark.read.jdbc(
    url=jdbc_url,
    table=select_query,
    properties=connection_details
)

display(source_df)

In [0]:
from delta.tables import DeltaTable
scd2_table_path = 's3://prudhvi-test-destination-02272025/customer/'
deltaTable = DeltaTable.forPath(spark, scd2_table_path)

In [0]:
%sql
select * from lakehouse.test.customer order by id asc

In [0]:
# Perform Merge (SCD2 Logic)
from pyspark.sql.functions import col, lit, current_date

# Perform Merge (SCD2 Logic)
merge_condition = "target.Id = source.Id AND target.current_status = 1"

update_condition = """
    target.firstname <> source.firstname OR 
    target.lastname <> source.lastname OR
    target.effectiveDate <> source.effectiveDate OR
    target.updateon <> source.updateon OR
    target.address <> source.address
"""

# Step 1: Update old record (if changed)
deltaTable.alias("target").merge(
    source_df.alias("source"),
    merge_condition
).whenMatchedUpdate(
    condition=update_condition,
    set={
        "endDate": current_date(),
        "current_status": lit(0)  # Mark the old record as inactive
    }
).execute()

# Step 2: Insert new version of updated records
new_records = source_df.alias("source").join(
    deltaTable.toDF().alias("target"),
    "Id",
    "inner"
).filter(update_condition).select(
    col("source.Id"),
    col("source.firstname"),
    col("source.lastname"),
    col("source.effectiveDate"),
    col("source.updateon"),
    col("source.address"),
    lit(None).cast("date").alias("endDate"),
    lit(1).alias("current_status")
)

new_records.write.format("delta").mode("append").save(scd2_table_path)


In [0]:
# Step 3: Insert completely new records
deltaTable.alias("target").merge(
    source_df.alias("source"),
    merge_condition
).whenNotMatchedInsert(
    values={
        "Id": col("source.Id"),
        "firstname": col("source.firstname"),
        "lastname": col("source.lastname"),
        "effectiveDate": col("source.effectiveDate"),
        "updateon": col("source.updateon"),
        "address": col("source.address"),
        "endDate": lit(None).cast("date"),
        "current_status": lit(1)
    }
).execute()

In [0]:
from pyspark.sql.functions import col,explode,max
rows = source_df.select(max('updateon').alias('max_date_time')).collect()
max_timestamp = rows[0][0]
print(max_timestamp)

In [0]:
if table_found:
    with engine.connect() as connection:
        table_name = "employee.tables_info"
        connection.execute(text("update {} set next_run_time='{}' where table_name='{}'".format(table_name,max_timestamp,'customer')))
        connection.commit()